# PCA

![pca2d](PCA_2d.svg)
![pca](pca.png)

## Analisis de Satisfacción de Clientes

Una encuesta para evaluar 20 centros sanitarios diferentes. 200 clientes evaluaron, en una escala del 1 al 10, cada una de las seis características del servicio:

1. Cortesía
2. Claridad
3. Competencia
4. Estado del centro
5. Rapidez del servicio
6. Horario de atención

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 

In [ ]:
df2 = pd.read_csv("csat.csv")
print(df2.head(10))

df2.describe()


In [ ]:
%matplotlib inline
df2.boxplot()


In [ ]:
#dataset scaling y visualizar
from sklearn.preprocessing import StandardScaler
scaler2 = StandardScaler(copy=False) 
scaler2.fit(df2.astype(float)) # 
scaler2.transform(df2.astype(float))
df2.boxplot()

In [ ]:
df2_scaled=pd.DataFrame(scaler2.transform(df2.astype(float))) 
df2_scaled.columns=df2.columns
df2_scaled.head()

In [ ]:
df2_scaled.boxplot()

In [ ]:
df2.apply(lambda s: df2.corrwith(s))

In [ ]:
#PCA fit
from sklearn.decomposition import PCA
# Podemos elegir el número de componentes, por ejemplo, 10, el porcentaje de la varianza total o establecerlo en None 
# (esto significa que elige automáticamente el número de componentes).
pca2 = PCA()
pca2.fit(df2_scaled) #El ajuste aprende algunas cantidades de los datos, las más importantes son los "componentes" y la "varianza explicada".

In [ ]:
#Usemos el pca para transformar el conjunto de datos
df2_pca = pd.DataFrame(pca2.transform(df2_scaled))
df2_pca

In [ ]:
#Analicemos lo sucedido
#VISUALICE La cantidad de varianza explicada por cada uno de los 10 componentes principales seleccionados.
pd.DataFrame(pca2.explained_variance_).transpose()

In [ ]:
#VISUALIZAR El porcentaje de varianza explicado por cada uno de los componentes seleccionados.
explained_var=pd.DataFrame(pca2.explained_variance_ratio_).transpose()
explained_var

In [ ]:
#VISUALIZAR El porcentaje acumulado de varianza explicada
cum_explained_var=np.cumsum(pca2.explained_variance_ratio_)
pd.DataFrame(cum_explained_var).transpose()

In [ ]:
%matplotlib inline
import seaborn as sns
ax = sns.barplot( data=explained_var)

In [ ]:
pd.DataFrame(pca2.components_,index=['PC1','PC2','PC3','PC4','PC5','PC6'],columns=df2.columns)


In [ ]:

def myplot(score,coeff,labels=None):
    xs = score[:,0]
    ys = score[:,1]
    n = coeff.shape[0]
    scalex = 1.0/(xs.max() - xs.min())
    scaley = 1.0/(ys.max() - ys.min())
    plt.scatter(xs * scalex,ys * scaley)
    for i in range(n):
        plt.arrow(0, 0, coeff[i,0], coeff[i,1],color = 'r',alpha = 0.5)
        if labels is None:
            plt.text(coeff[i,0]* 1.15, coeff[i,1] * 1.15, "Var"+str(i+1), color = 'g', ha = 'center', va = 'center')
        else:
            plt.text(coeff[i,0]* 1.15, coeff[i,1] * 1.15, labels[i], color = 'g', ha = 'center', va = 'center')
    plt.xlim(-1,1)
    plt.ylim(-1,1)
    plt.xlabel("PC{}".format(1))
    plt.ylabel("PC{}".format(2))
    plt.grid()

#Call the function. Use only the 2 PCs.
myplot(pca2.transform(df2_scaled)[:,0:2],np.transpose(pca2.components_[0:2, :]), df2.columns)
plt.show()

### 1° componente:

Las variables

- Estado
- Puntualidad
- Horario de atención

muestran una alta correlación con el primer componente. Este componente puede resumirse como un índice del **desempeño de la estructura**.

### 2º componente:

Las variables
- Cortesía
- Claridad
- Competencia

muestran una alta correlación con el segundo componente.
Este componente puede resumirse como un índice del **desempeño del personal**.

**Observe que los Componentes Principales tienen valores negativos en las variables que explican.**

In [ ]:
df2_pca.columns=['PC1','PC2','PC3','PC4','PC5','PC6']
df2_pca

In [ ]:
p1=sns.scatterplot(x="PC1", y="PC2",
              alpha=.3, 
              hue="PC6", legend=False,
              data=df2_pca);

# agrega anotaciones una por una con un bucle
for line in range(0,df2_pca.shape[0]):
     p1.text(df2_pca.PC1[line], df2_pca.PC2[line], line, horizontalalignment='left', size='medium', color='black')


Concluimos que:
- Los centros 18 y 11 tienen una BUENA infraestructura, pero un servicio MALO.
- El grupo cerca de los centros 1, 4 y 6 tiene una MALA infraestructura, pero una BUENA calidad de servicio.
- Los grupos 0, 3, 2 y 9 tienen una BUENA infraestructura y servicio.
- El centro 17 tiene una MALA infraestructura y servicio.

## Cancer de mamas (diagnostic) dataset

In [ ]:
#Subir conjuntos de datos de juguetes desde scikit-learn
#sklearn incluye algunos conjuntos de datos estándar pequeños que no requieren la descarga de ningún archivo de un sitio web externo.
from sklearn.datasets import load_breast_cancer
dataset = load_breast_cancer() #El conjunto de datos del cáncer de mama es un conjunto de datos de clasificación binaria clásico y muy sencillo.

#crea el dataframe
dataset_df = pd.DataFrame(dataset.data)
columns = dataset.feature_names
dataset_df.columns = columns

print(dataset["DESCR"])

In [ ]:
dataset_df


In [ ]:
#dataset visualization tools
%matplotlib inline
dataset_df.boxplot()
dataset_df.head()

In [ ]:
#dataset scaling y visualización
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler(copy=False) #o alternativamente usar MinMaxScaler
scaler.fit(dataset_df) 
scaler.transform(dataset_df) 
dataset_df.boxplot()
dataset_df.head()

In [ ]:
dataset_df.columns

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = dataset_df.loc[:,'mean radius']
y = dataset_df.loc[:,'worst symmetry']


plt.scatter(x, y,alpha=0.2,c=dataset.target )
plt.show()

In [ ]:
%matplotlib inline
import seaborn as sns

df_sample = dataset_df.copy()
df_sample = df_sample.iloc[:,:8]
df_sample['target']=dataset.target
sns.pairplot(df_sample, hue='target')

In [ ]:
#PCA fit
from sklearn.decomposition import PCA
# podemos elegir el número de componentes, por ejemplo 10, el porcentaje de la varianza total o establecerlo en None
#(eso significa que elige automáticamente el número de componentes)
pca = PCA(n_components=10)
pca.fit(dataset_df) #El ajuste aprende algunas cantidades de los datos, las más importantes son los "componentes" y la "varianza explicada".

In [ ]:
#Usemos pca para transformar el conjunto de datos
x_pca = pca.transform(dataset_df)
print("Dataset shape before PCA: ", dataset_df.shape)
print("Dataset shape after PCA: ", x_pca.shape)

In [ ]:
#Analicemos lo sucedido
#VISUALICE La cantidad de varianza explicada por cada uno de los 10 componentes principales seleccionados.
pd.DataFrame(pca.explained_variance_).transpose()

In [ ]:
#VISUALIZAR El porcentaje de varianza explicado por cada uno de los componentes seleccionados.
explained_var=pd.DataFrame(pca.explained_variance_ratio_).transpose()
explained_var

In [ ]:
%matplotlib inline
import seaborn as sns
sns.barplot( data=explained_var)

In [ ]:
#VISUALIZAR El porcentaje acumulado de varianza explicada
cum_explained_var = np.cumsum(pca.explained_variance_ratio_)
pd.DataFrame(cum_explained_var).transpose()

In [ ]:
#Print el porcentaje total de varianza explicada
print(cum_explained_var[-1]) 

In [ ]:
#ELECCIÓN DEL NÚMERO DE COMPONENTES - podemos graficar el porcentaje acumulado de varianza explicada
import matplotlib.pyplot as plt
plt.plot(cum_explained_var)
plt.xlabel('number of components')
plt.ylabel('cumulative explained variance');

Esta curva anterior cuantifica qué proporción de la varianza total de 30 dimensiones se encuentra en los primeros 10 componentes.
Por ejemplo, vemos que los primeros 4 componentes contienen aproximadamente el 79 % de la varianza, mientras que se necesitan alrededor de 6 componentes para describir cerca del 95 % de la varianza.

In [ ]:
pd.DataFrame(pca.components_,index=['pc1', 'pc2','pc3','pc4','pc5', 'pc6','pc7','pc8','pc9','pc10'],columns=dataset_df.columns)

In [ ]:
# Veamos las coordenadas de los datos en la PCA.
principalDf = pd.DataFrame(data = x_pca
             , columns = ['pc1', 'pc2','pc3','pc4','pc5', 'pc6','pc7','pc8','pc9','pc10'])
principalDf.head()

In [ ]:
principalDf['target']=dataset.target
dataset.target_names

In [ ]:
# Los datos en el primer PCA
sns.scatterplot(x="pc1",y=[0]*(principalDf['target'].size),
              hue="target", alpha=.2,
              data =principalDf);

In [ ]:
# Los datos en el segundo PCA
sns.scatterplot(x="pc1", y="pc2",
              hue="target", alpha=.3,
              data=principalDf);

In [ ]:
# La data en los tres PCA 

from mpl_toolkits.mplot3d import axes3d

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(principalDf['pc1'], principalDf['pc2'],principalDf['pc3'], c=principalDf['target'], s=40)
ax.view_init(60, 60)
plt.show()

In [ ]:
#Además de elegir explícitamente el número de componentes principales, se pueden usar herramientas automáticas como:

#(1) Puede dejar que la implementación de PCA de sklearn elija el número de componentes usando:
#Establezca n_components == 'mle' y svd_solver == 'full' y utilice el MLE de Minka para calcular la dimensión.
    
pca = PCA(n_components='mle',svd_solver='full') 
pca.fit(dataset_df)
pca.n_components_ 
#y luego transformar el conjunto de datos como ya hemos visto anteriormente

In [ ]:
#Usemos pca para transformar el conjunto de datos
x_pca = pca.transform(dataset_df)
print("Dataset shape before PCA: ", dataset_df.shape)
print("Dataset shape after PCA: ", x_pca.shape)

In [ ]:
#O (2) puede solicitar los componentes capaces de explicar un cierto porcentaje de varianza mediante:
#Establezca 0 < n_components < 1 y svd_solver == 'full' para seleccionar el número de componentes de manera que
# la cantidad de varianza que se debe explicar sea mayor que el porcentaje especificado por n_components.

pca = PCA(n_components=0.9,svd_solver='full') 
pca.fit(dataset_df)
pca.n_components_ 
#y luego transformar el conjunto de datos como ya hemos visto anteriormente

In [ ]:
#Usemos pca para transformar el conjunto de datos
x_pca = pca.transform(dataset_df)
print("Dataset shape before PCA: ", dataset_df.shape)
print("Dataset shape after PCA: ", x_pca.shape)

## mtcars

In [ ]:
# Import CSV mtcars
cars = pd.read_csv('mtcars.csv',index_col = 0)

The car types are a mix that includes sedans (Datsun, Ford, Honda,…), luxury sedans (Mercedes, Cadellac,..), muscle cars (Javelin, Challenger, Camaro…) and high-end sports cars (Porsche, Lotus, Maserati, Ferrari…)

- 	mpg 	Miles/US Gallon 	mpg is the determinant of fuel efficiency
- 	cyl 	Number of cylinders 	Data includes vehicles with 4,6,8 cylinder engines.
- 	disp 	Displacement (cu.in.) 	Displacement measures overall volume in the engine as a factor of cylinder circumfrance, depth and total number of cylinders. This metric gives a good proxy for the total amount of power the engine can generate.
- 	hp 	Gross horsepower 	Gross horsepower measures the theoretical output of an engine’s power output
- 	drat 	Rear axle ratio 	The rear axle gear ratio indicates the number of turns of the drive shaft for every one rotation of the wheel axle. 
-   wt      weigth of the car 
- 	qsec 	1/4 mile time 	A performance measure, primarily of acceleration. Fastest time to travel 1/4 mile from standstill (in seconds).
- 	vs 	V/S 	Binary variable signaling the engine cylinder configuration a V-shape (vs=0) or Straight Line (vs=1). V==0 and S==1. 
- 	am 	Transmission Type 	A binary variable signaling whether vehicle has automatic (am=0) or manual (am=1) transmission configuration.
- 	gear 	Number of forward gears 	Number of gears in the transmission.
- 	carb 	Number of carburetors 	The number of carburetor barrels.

In [ ]:
cars.head()

In [ ]:
#dataset scaling and visualizing
from sklearn.preprocessing import StandardScaler
scaler3 = StandardScaler(copy=False) #or alternatively use MinMaxScaler
scaler3.fit(cars.astype(float)) 
df_cars=pd.DataFrame(scaler3.transform(cars.astype(float))) 
df_cars.columns=cars.columns
df_cars.boxplot()

In [ ]:
from sklearn.decomposition import PCA
pca3 = PCA()
pca3.fit(df_cars)

In [ ]:
explained = pd.DataFrame(pca3.explained_variance_ratio_).transpose()
sns.barplot(data=explained)

In [ ]:
pd.DataFrame(pca3.components_,columns=cars.columns)

In [ ]:
cars_pca = pd.DataFrame(pca3.transform(df_cars),columns = ['pc1', 'pc2','pc3','pc4','pc5', 'pc6','pc7','pc8','pc9','pc10','pc11']
                        ,index=cars.index.values)
cars_pca.head()

In [ ]:
p2=sns.scatterplot(x="pc1", y="pc2",
              alpha=.3,
              data=cars_pca);
# add annotations one by one with a loop
for line in range(0,cars_pca.shape[0]):
     p2.text(cars_pca.pc1[line], cars_pca.pc2[line], cars_pca.index[line], horizontalalignment='left', size='medium', color='black')


## Iris database

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import datasets

iris = datasets.load_iris()


In [ ]:
print(iris['DESCR'])

In [ ]:
#create the dataframe
iris_df = pd.DataFrame(iris.data)
iris_df.columns = iris.feature_names

In [ ]:
from sklearn.preprocessing import StandardScaler
iris_scaler = StandardScaler(copy=False) #or alternatively use MinMaxScaler
iris_scaler.fit(iris_df) # 
iris_scaler.transform(iris_df)
iris_df.boxplot()

In [ ]:
#PCA fit
from sklearn.decomposition import PCA
pca = PCA()
pca.fit(iris_df)

In [ ]:
#VISUALIZE The percentage of variance explained by each of the selected components.
pd.DataFrame(pca.explained_variance_ratio_).transpose()


In [ ]:
explained = pd.DataFrame(pca.explained_variance_ratio_).transpose()
sns.barplot(data=explained)

In [ ]:
pd.DataFrame(pca.components_,columns=iris_df.columns)

In [ ]:
# Let see the coordinates of the data in the PCA 
iris_pca = pd.DataFrame(pca.transform(iris_df),columns = ['pc1', 'pc2','pc3','pc4']
                        ,index=iris_df.index.values)
iris_pca 


In [ ]:
# The data in the first PCA 
sns.scatterplot(x="pc1", y=0,
              hue=iris['target'], alpha=.8,
              data=iris_pca,
               palette="deep");

In [ ]:
# The data in the first two PCA 
sns.scatterplot(x="pc1", y="pc2",
              hue=iris['target'], alpha=.3,
              data=iris_pca);